In [ ]:
from analysis_helper import *

import warnings
from pathlib import Path
from matplotlib.patches import Patch

savefig = False

# Extract all sensitivities

In [ ]:
# conference_v1 scenarios
# scenarios = [
#     "baseline-aims-3H", "decarbonize-aims-3H",
#     "baseline-asean-3H", "decarbonize-asean-3H",
#     "baseline-existing-3H", "decarbonize-existing-3H", 
# ]
# years = [2025, 2030, 2035, 2040, 2045, 2050]

# sensitivity scenarios
scenarios = [
    "decarbonize-aims-3H",
    "sensitivity-roof100-3H",
    "sensitivity-roof200-3H",
    "sensitivity-1H",
    "sensitivity-3H",
    "sensitivity-6H",
    "sensitivity-50n-3H",
    "sensitivity-200n-3H",
]
years = [2050]


df_scene = pd.DataFrame(index=scenarios, columns=years)
for scenario in df_scene.index:
    for year in df_scene.columns:
        base_path = Path(f"../results/{scenario}/postnetworks/")
        try:
            fn = [p for p in base_path.rglob("*.nc") if str(year) in p.name][0]
            n = pypsa.Network(fn)
            n = clean_carrier(n)
            df_scene.loc[scenario,year] = n
        except:
            df_scene.loc[scenario,year] = np.nan
            break # Break one level loop

# Generate Sensitivity Deltas

In [ ]:
def plot_scenario_generation_ax(df, year, ax, compare=False):

    scenarios = df.columns
    
    # ---- Data aggregation ----
    df_energy_all = pd.DataFrame(columns=scenarios)
    df_load_all = pd.DataFrame(columns=scenarios)
    df_store_all = pd.DataFrame(columns=scenarios)
    
    for scenario in scenarios:
        n = df.loc[year, scenario]
    
        df_energy = pd.DataFrame(
            n.statistics.energy_balance(
                comps=['Generator', 'Link', 'StorageUnit', 'Store'],
                nice_names=False
            )
        ).reset_index()
    
        df_energy = df_energy[
            ~df_energy.carrier.isin(elec_bus_carrier) &
            df_energy.bus_carrier.isin(elec_bus_carrier)
        ]
    
        df_energy_all[scenario] = df_energy.groupby("carrier").sum()[0]
    
        df_load_all.loc["Demand", scenario] = -n.statistics.energy_balance(
            comps=["Load"]
        ).sum() / 1e6
    
        store_bus = n.stores[n.stores.carrier.isin(
            ['H2 Store Tank', 'battery', 'home battery']
        )].index
    
        df_store_all.loc["Storage Discharge", scenario] = (
            - n.snapshot_weightings.objective @ n.stores_t.p[store_bus].clip(upper=0)
        ).sum() / 1e6
    
    df_energy_all = df_energy_all.rename(index=battery_pair).groupby("carrier").sum() / 1e6
    
    df_pos = df_energy_all.T.clip(lower=0)
    df_pos = df_pos.loc[:, df_pos.sum(skipna=True) > 1]
    
    ordered_carrier = [c for c in order_carrier if c in df_pos.columns]
    remaining_cols = [c for c in df_pos.columns if c not in order_carrier]
    df_pos = df_pos.reindex(columns=ordered_carrier + remaining_cols)

    if compare:
        df_pos -= df_pos.loc[compare]
        df_pos = df_pos.drop(index=compare)

        df_store_all = df_store_all.T
        df_store_all -= df_store_all.loc[compare]
        df_store_all = df_store_all.drop(index=compare)
        df_store_all = df_store_all.T

        ax.axhline(0, color='black', linewidth=1)
    
    # Generation area
    df_pos.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[n.carriers.color[c] for c in df_pos.columns],
        linewidth=0,
        alpha=0.8,
        legend=False
    )
    
    # Storage discharge
    df_store_all.T.plot(
        kind="bar",
        ax=ax,
        linewidth=2,
        facecolor='none',
        edgecolor='blue',
        linestyle='--',
        alpha=0.8,
        legend=False
    )
    
    # # Load demand
    if not compare:
        df_load_all.T.plot(
            kind="bar",
            ax=ax,
            linewidth=2,
            facecolor='none',
            edgecolor='black',
            linestyle='--',
            alpha=0.8,
            legend=False
        )
    
    ax.set_ylabel("Δ Energy [TWh]")
    ax.grid(axis='y')

    return ax

def plot_scenario_costs_ax(df, year, ax, compare=False):

    # --- Rename carriers ---
    mapping = {
        ('Generator', 'lignite'): ('Generator', 'lignite fuel'),
        ('Generator', 'coal'): ('Generator', 'coal fuel'),
        ('Generator', 'gas'): ('Generator', 'gas fuel'),
    }
    
    hatched_carriers = {"lignite fuel", "coal fuel", "gas fuel", "solid biomass"}

    scenarios = df.columns
    
    # ---- Data aggregation ----
    df_system_cost_all = pd.DataFrame(columns=scenarios)
    
    for scenario in scenarios:
        n = df.loc[year, scenario]
    
        costs = n.statistics.system_cost(nice_names=False)
        costs.index = costs.index.map(lambda x: mapping.get(x, x))
    
        df_system_cost_all[scenario] = costs.groupby("carrier").sum()
    
    df_system_cost_all = (
        df_system_cost_all
        .rename(index=battery_pair)
        .groupby("carrier")
        .sum() / 1e9
    )
    
    # ---- Prepare plotting data ----
    df_pos = df_system_cost_all.T.clip(lower=0)
    df_pos = df_pos.loc[:, df_pos.sum(skipna=True) > 1]
    
    ordered_carrier = [c for c in order_carrier if c in df_pos.columns]
    remaining_cols = [c for c in df_pos.columns if c not in order_carrier]
    df_pos = df_pos.reindex(columns=ordered_carrier + remaining_cols)

    if compare:
        df_pos -= df_pos.loc[compare]
        df_pos = df_pos.drop(index=compare)

        ax.axhline(0, color='black', linewidth=1)
    
    # ---- Plot ----
    df_pos.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[n.carriers.color[c] for c in df_pos.columns],
        linewidth=0,
        alpha=0.8,
        legend=False,
    )
    
    # ---- Apply hatching ----
    for container, carrier in zip(ax.containers, df_pos.columns):
        if carrier in hatched_carriers:
            for patch in container:
                patch.set_hatch("//")      # use double slash for visibility
                patch.set_edgecolor("k")
                patch.set_linewidth(0.3)

    ax.set_ylabel("Δ System Cost [Billion €/a]")
    ax.grid(axis='y')

    return ax

def combined_figure_legend(fig, loc="upper center", ncol=1, **legend_kwargs):
    """
    Create a combined legend for all axes in a figure without duplicate labels.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure containing multiple axes.
    loc : str, optional
        Legend location.
    ncol : int, optional
        Number of legend columns.
    legend_kwargs : dict
        Additional keyword arguments passed to fig.legend().
    """
    from collections import OrderedDict
    
    handles, labels = [], []

    # Collect legend entries from all axes
    for ax in fig.axes:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)

    # De-duplicate while preserving first occurrence
    unique = OrderedDict(zip(labels, handles))

    # Split into ordered and unordered labels
    ordered_handles = []
    ordered_labels = []

    for label in order_carrier:
        if label in unique:
            ordered_labels.append(label)
            ordered_handles.append(unique[label])

    # Append labels not in order_carrier
    for label, handle in unique.items():
        if label not in order_carrier:
            ordered_labels.append(label)
            ordered_handles.append(handle)

    ordered_handles = ordered_handles[::-1]
    ordered_labels = ordered_labels[::-1]

    nice_name = n.carriers.nice_name.map(nice_name_rename).fillna(n.carriers.nice_name)
    ordered_labels = [nice_name[lable] if lable in nice_name else lable for lable in ordered_labels]

    fig.legend(
        ordered_handles,
        ordered_labels,
        title="Carrier",
        loc=loc,
        ncol=ncol,
        bbox_to_anchor=(0.9, 0.45),
        **legend_kwargs
    )

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6, 6), sharex=True)

year = 2050
df = df_scene.T.copy(deep=True)
scenario_compare = "sensitivity-"
compare = "3H"
df.columns = df.columns.str.replace(scenario_compare, "", regex=False)
df.columns = df.columns.str.replace("decarbonize-aims", "myopic", regex=False)

#for ax, scenario in zip(axes.flat, df_scene.index):
plot_scenario_generation_ax(df, year, ax=axes[0], compare=compare)
plot_scenario_costs_ax(df, year, ax=axes[1], compare=compare)

combined_figure_legend(fig, loc="center left", ncol=1)

axes[0].set_title(f"Sensitivity Analysis relative to overnight-{compare}")
axes[0].set_ylim([-420, 420])
axes[1].set_ylim([-32, 32])

if savefig:
    fig.savefig(
        f"sensitivity-{compare}.png",
        dpi=300,
        bbox_inches="tight",
    )

plt.show()